<a href="https://colab.research.google.com/github/nishthadighe-bit/Data--Engineering-Practicals/blob/main/Practical_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

# =====================================================================
# STEP 1: INITIALIZE TARGET DATA WAREHOUSE & SCHEMA
# =====================================================================
db_name = "production_warehouse.db"
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# Create Warehouse Table with Constraints
cursor.execute("DROP TABLE IF EXISTS target_orders")
cursor.execute("""
    CREATE TABLE target_orders (
        order_id INTEGER PRIMARY KEY,
        customer_name TEXT NOT NULL,
        order_amount REAL NOT NULL,
        order_date TEXT NOT NULL,
        loaded_at TEXT NOT NULL
    )
""")
conn.commit()

# Insert Existing Historical Batch (Initial Load)
initial_records = [
    (1001, 'Aarav Mehta', 450.00, '2026-09-10', '2026-09-10 10:00:00'),
    (1002, 'Riya Sharma', 1200.50, '2026-09-11', '2026-09-11 10:00:00')
]
cursor.executemany("INSERT INTO target_orders VALUES (?,?,?,?,?)", initial_records)
conn.commit()

print("--- Initial Target Warehouse Data ---")
print(pd.read_sql_query("SELECT * FROM target_orders", conn))

# =====================================================================
# STEP 2: INCOMING INCREMENTAL BATCH (With Quality Issues)
# =====================================================================
incoming_batch = pd.DataFrame([
    {"order_id": 1002, "customer_name": "Riya Sharma", "order_amount": 1200.50, "order_date": "2026-09-11"}, # Existing duplicate
    {"order_id": 1003, "customer_name": "Karan Patel", "order_amount": 890.00, "order_date": "2026-09-14"},  # Valid new record
    {"order_id": 1004, "customer_name": "Neha Verma", "order_amount": -150.00, "order_date": "2026-09-14"},  # Quality Error: Negative amount
    {"order_id": 1005, "customer_name": None, "order_amount": 310.00, "order_date": "2026-09-15"},           # Quality Error: Missing name
    {"order_id": 1006, "customer_name": "Vikram Das", "order_amount": 1750.25, "order_date": "2026-09-15"}   # Valid new record
])

print("\n--- Incoming Incremental Batch ---")
print(incoming_batch)

# =====================================================================
# STEP 3: AUTOMATED DATA QUALITY VALIDATION CHECKS
# =====================================================================
print("\n--- Running Data Quality Validation Suite ---")

# Rule 1: Schema & Missing Value Check
valid_records = incoming_batch.dropna(subset=['order_id', 'customer_name', 'order_amount']).copy()

# Rule 2: Non-negative Numerical Check
valid_records = valid_records[valid_records['order_amount'] > 0]

# Rule 3: Data Type Alignment
valid_records['order_id'] = valid_records['order_id'].astype(int)
valid_records['order_amount'] = valid_records['order_amount'].astype(float)

print(f"Passed Quality Checks: {len(valid_records)} out of {len(incoming_batch)} records.")

# =====================================================================
# STEP 4: INCREMENTAL LOADING (Delta Load via Key Filtering)
# =====================================================================
print("\n--- Performing Incremental Loading (Deduplication) ---")

# Fetch existing keys from Data Warehouse
existing_ids = pd.read_sql_query("SELECT order_id FROM target_orders", conn)['order_id'].tolist()

# Filter out records that already exist in target DB
incremental_delta = valid_records[~valid_records['order_id'].isin(existing_ids)].copy()

if not incremental_delta.empty:
    # Add audit timestamp metadata
    incremental_delta['loaded_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Load Delta into Warehouse
    incremental_delta.to_sql("target_orders", conn, if_exists="append", index=False)
    print(f"Successfully loaded {len(incremental_delta)} new incremental records!")
else:
    print("No new records to load.")

# =====================================================================
# STEP 5: FINAL WAREHOUSE AUDIT VERIFICATION
# =====================================================================
print("\n--- Final Data Warehouse Table Contents ---")
final_df = pd.read_sql_query("SELECT * FROM target_orders", conn)
print(final_df)

conn.close()
print("\nPractical 8 Quality Assurance & Incremental Pipeline Executed Successfully!")

--- Initial Target Warehouse Data ---
   order_id customer_name  order_amount  order_date            loaded_at
0      1001   Aarav Mehta         450.0  2026-09-10  2026-09-10 10:00:00
1      1002   Riya Sharma        1200.5  2026-09-11  2026-09-11 10:00:00

--- Incoming Incremental Batch ---
   order_id customer_name  order_amount  order_date
0      1002   Riya Sharma       1200.50  2026-09-11
1      1003   Karan Patel        890.00  2026-09-14
2      1004    Neha Verma       -150.00  2026-09-14
3      1005          None        310.00  2026-09-15
4      1006    Vikram Das       1750.25  2026-09-15

--- Running Data Quality Validation Suite ---
Passed Quality Checks: 3 out of 5 records.

--- Performing Incremental Loading (Deduplication) ---
Successfully loaded 2 new incremental records!

--- Final Data Warehouse Table Contents ---
   order_id customer_name  order_amount  order_date            loaded_at
0      1001   Aarav Mehta        450.00  2026-09-10  2026-09-10 10:00:00
1      1002